# Tutorial - Active Magnetic Bearings (AMB)


This tutorial provides a comprehensive guide on how to model, simulate, and analyze Active Magnetic Bearings (AMBs) using ROSS.

# Section 1: Introduction and Working Principle

**Active Magnetic Bearings (AMBs)** are support mechanisms that levitate a rotating shaft using electromagnetic forces, eliminating physical contact between the rotor and the stator.

## 1.1 Working Principle

An AMB system operates as a closed-loop feedback control system consisting of three main components:

1. **Sensors:** Measure the radial displacement of the rotor ($x$).
2. **Controller:** Processes the displacement signal and calculates the necessary current correction ($i$) to maintain the rotor's position (setpoint).
3. **Actuators (Electromagnets):** Receive the current and generate the magnetic force required to pull the rotor back to the center.

## 1.2 Mathematical Model

In ROSS, the electromagnetic force $F$ is linearized around a nominal operating point (bias current $i_0$ and nominal gap $g_0$). The force equation is generally described by:

$$
 F = K_i \; i - K_s \; x 
$$

Where:

- $K_i$ is the **current stiffness** (Force/Current factor).
- $K_s$ is the **negative electromagnetic stiffness** (Force/Displacement factor), representing the inherent instability of the magnetic attraction. Following ISO 14839-1, $K_s$ is negative and is treated as a stiffness, so the term $-K_s \, x$ has the same sign as $x$: the rotor is pulled toward the closer pole. The same $K_s$ enters the rotor stiffness matrix in the frequency-domain analyses.

# Section 2: Declaring a Magnetic Bearing

To use an AMB in ROSS, you must instantiate the `MagneticBearingElement` class. You need to provide physical parameters of the electromagnet and the gains for the PID controller.

The actuator node is defined by `n`, while `sensor_node` defines the node where the rotor displacement is measured. When `sensor_node` is omitted, it defaults to `n`, resulting in a collocated sensor–actuator configuration. The property `is_non_collocated` reports whether the current sensor and actuator nodes are different.

In [1]:
import ross as rs
import numpy as np

# Physical parameters of the actuator
n_node = 4  # Node where the bearing is located
g0 = 1e-3  # Nominal air gap (m)
i0 = 1.0  # Bias current (A)
ag = 1e-4  # Pole area (m²)
nw = 200  # Number of windings
alpha = np.pi / 8  # Pole angle (rad)

# PID Controller gains
Kp = 1500  # Proportional gain
Kd = 10  # Derivative gain
Ki = 100  # Integral gain

# Instantiation
amb = rs.MagneticBearingElement(
    n=n_node,
    sensor_node=n_node,
    g0=g0,
    i0=i0,
    ag=ag,
    nw=nw,
    alpha=alpha,
    kp_pid=Kp,
    kd_pid=Kd,
    ki_pid=Ki,
    tag="AMB_1",
)

import plotly.io as pio

pio.renderers.default = "notebook_connected"

# Section 3: Modal Sensor–Actuator Non-Collocation Analysis

The sensor and actuator positions defined in the previous section determine how each rotor mode is measured and controlled.

When the displacement sensor and the magnetic actuator are located at different axial positions, the system presents sensor–actuator
non-collocation. In this case, the displacement measured by the controller may have a different amplitude or sign from the displacement
at the actuator location.

This section introduces a modal-residue analysis that can be used to screen candidate sensor positions for a selected magnetic bearing.

## 3.1 Modal basis

For a linear rotor model, the lateral displacement projected in a direction $\theta$ can be approximated by a modal expansion:

$$
q^{(\theta)}(x,t)
\approx
\sum_{r=1}^{n_m}
\phi_r^{(\theta)}(x)\eta_r(t),
$$

where $\phi_r^{(\theta)}(x)$ is the projected shape of mode $r$ and $\eta_r(t)$ is its modal coordinate.

The projected mode shape is obtained from the lateral components as

$$
\phi_r^{(\theta)}(x)
=
\cos(\theta)\phi_{x,r}(x)
+
\sin(\theta)\phi_{y,r}(x).
$$

For an actuator located at $x_a$ and a sensor located at $x_s$, the contribution of mode $r$ to the sensor–actuator transfer relation is
proportional to

$$
R_r(x_s)
=
\phi_r^{(\theta)}(x_a)
\phi_r^{(\theta)}(x_s).
$$

The quantity $R_r(x_s)$ is referred to here as the modal residue. It combines the modal participation at the actuator with the modal
observability at the sensor.

For a classical modal representation, the corresponding transfer relation may be written in the form

$$
G_{sa}(s)
\approx
\sum_{r=1}^{n_m}
\frac{
    \phi_r^{(\theta)}(x_s)
    \phi_r^{(\theta)}(x_a)
}{
    m_r
    \left(
        s^2
        + 2\zeta_r\omega_r s
        + \omega_r^2
    \right)
},
$$

where $m_r$, $\zeta_r$, and $\omega_r$ are the modal mass, damping ratio, and natural frequency of mode $r$, respectively.

Therefore, the product of the modal amplitudes at the sensor and actuator positions determines the sign and relative participation of
that mode in the transfer relation.

Because the absolute magnitude of a mode shape depends on its scaling,
ROSS normalizes the residues independently for each retained mode:

$$
\overline{R}_r(x_s)
=
\frac{
    R_r(x_s)
}{
    \displaystyle
    \max_{x_j}
    \left|R_r(x_j)\right|
}.
$$

For a selected tolerance $\tau$, the sensor positions are classified as

$$
C_r(x_s)
=
\begin{cases}
+1, & \overline{R}_r(x_s) > \tau, \\[4pt]
0,  & \left|\overline{R}_r(x_s)\right| \leq \tau, \\[4pt]
-1, & \overline{R}_r(x_s) < -\tau.
\end{cases}
$$

A positive residue indicates that the sensor and actuator have the same modal sign. A negative residue indicates opposite modal signs. A value close to zero indicates that the sensor or actuator has low participation in that mode, usually because it is close to a modal node.

A sensor close to a modal node may have low observability for that mode, while an actuator close to a modal node may have low control authority. The sign of the residue also determines the sign of the corresponding modal contribution to the sensor–actuator transfer relation.

> **Important:** The terms *same modal sign*, *opposite modal signs*,
> and *near modal node* describe only the modal relationship between the
> selected positions. They do not, by themselves, guarantee closed-loop
> stability or satisfactory controller performance. A complete control
> assessment must also consider the controller dynamics, structural
> damping, operating speed, actuator limitations, and the interaction
> among the retained modes.

## 3.2 Example rotor model

The following example uses a flexible rotor supported by two active magnetic bearings. Each bearing may have an independent actuator and
sensor position.

The example model is created directly because no physical or controller parameters from the previous example need to be redefined.

In [2]:
rotor = rs.rotor_example_amb_general_controllers()

magnetic_bearings = [
    bearing
    for bearing in rotor.bearing_elements
    if isinstance(
        bearing,
        rs.MagneticBearingElement,
    )
]

rotor.plot_rotor().show()

## 3.3 Defining the sensor positions

The actuator nodes of the example model are kept unchanged. Only the measurement positions are modified, placing each sensor at a neighboring rotor node.

In [3]:
sensor_nodes_by_actuator = {
    11: 10,
    39: 40,
}

for bearing in magnetic_bearings:
    bearing.sensor_node = sensor_nodes_by_actuator[int(bearing.n)]

    print(
        f"{bearing.tag}: "
        f"actuator={bearing.n}, "
        f"sensor={bearing.sensor_node}, "
        f"non_collocated={bearing.is_non_collocated}"
    )

Magnetic Bearing 0: actuator=11, sensor=10, non_collocated=True
Magnetic Bearing 1: actuator=39, sensor=40, non_collocated=True


The dictionary relates each actuator node to its corresponding sensor node. In a user-defined rotor, the selected sensor nodes must belong to the rotor model.

The property `is_non_collocated` is evaluated independently for each magnetic bearing.

## 3.4 Running the analysis

The modal residue depends on the actuator location. Therefore, each magnetic bearing is analyzed independently.

The analyses below are calculated once and stored in a dictionary using the bearing tags as keys.

In [4]:
results_by_tag = {
    bearing.tag: rotor.run_amb_non_collocation(
        magnetic_bearing=bearing,
        speed=0.0,
        modes=range(8),
        direction="x",
        residue_tolerance=0.05,
    )
    for bearing in magnetic_bearings
}

The main arguments are:

| Argument | Description |
|---|---|
| `magnetic_bearing` | AMB whose actuator is used as the reference position. |
| `speed` | Rotor speed used in the modal analysis. The current implementation accepts only `0.0`. |
| `modes` | Zero-based indices of the requested modes. |
| `sensor_nodes` | Optional subset of candidate sensor nodes. When omitted, all rotor nodes are evaluated. |
| `direction` | Lateral projection direction: `"x"`, `"y"`, or an angle in radians. |
| `residue_tolerance` | Normalized interval treated as a near-modal-node region. |

Only lateral modes are retained. Requested torsional or axial modes are
excluded from the residue calculation while their indices, frequencies,
and mode types remain available in the result metadata.

## 3.5 Inspecting the modal-residue results

The first magnetic bearing is selected for a detailed inspection. The returned object contains the retained mode shapes, modal residues,
normalized residues, classifications, and metadata for the excluded modes.

The modal-residue summary is returned as a pandas `DataFrame`, which can be displayed directly in a notebook.

In [5]:
analyzed_bearing = magnetic_bearings[0]
results = results_by_tag[analyzed_bearing.tag]

table = results.modal_residue_table()
table

,Mode,Frequency (Hz),Actuator amplitude,Sensor amplitude,Modal residue,Normalized residue,Classification
0,1,8.708849,0.839213,0.850241,0.713533,0.850241,Same modal sign
1,2,8.708849,0.839213,0.850241,0.713533,0.850241,Same modal sign
2,3,10.958039,0.371135,0.400752,0.148733,0.400752,Same modal sign
3,4,10.958039,0.371135,0.400752,0.148733,0.400752,Same modal sign
4,5,107.050978,0.149177,0.207302,0.030925,0.207302,Same modal sign
5,6,107.051027,0.149177,0.207302,0.030925,0.207302,Same modal sign
6,7,401.918563,0.277647,0.192967,0.053577,0.192967,Same modal sign
7,8,401.918563,0.277647,0.192967,0.053577,0.192967,Same modal sign


The mode numbers displayed in the table preserve the original numbering of the requested modal solution. Consequently, excluded non-lateral modes are not silently renumbered.

The actuator and sensor amplitudes correspond to the projected and normalized mode shape. The modal residue is their product, and the
normalized residue is used for the classification.

If the results are being inspected outside a notebook, the same table can be printed with:

```python
print(table.to_string(index=False))
```

## 3.6 Interactive sensor-position map

The combined visualization is the default interface. It initially displays the sensor-position map, and a retained lateral mode can then be selected from the dropdown menu to display its projected mode shape below the map.

The analyzed magnetic bearing is highlighted, while the actuator and sensor positions of the other magnetic bearings are also shown as contextual information.

In [6]:
fig = results.plot()
fig.show()

When a mode is selected:

- the corresponding row remains highlighted in the sensor-position map;
- the other retained modes are shown with reduced emphasis;
- the projected mode shape is displayed below the map;
- the analyzed magnetic bearing is highlighted;
- the other magnetic bearings remain visible as contextual actuator-sensor pairs;
- a warning marker indicates low actuator participation when applicable.

## 3.7 Generating separate figures

The combined visualization is the default interface because it allows the sensor-position map and the selected mode shape to be inspected in a single interactive figure.

Alternatively, the map and mode shapes can be generated as independent figures by setting `combined=False`:

```python
separate_plots = results.plot(
    combined=False,)
```

When `combined=False`, the sensor-position map and all retained lateral mode-shape figures are displayed automatically. The method also returns a dictionary containing the generated figures:

- `"sensor_map"` contains the complete sensor-position map;
- `"mode_shapes"` contains one figure for each retained lateral mode.

The keys of `"mode_shapes"` correspond to the original zero-based modal indices. This allows a specific figure to be accessed later, for example:

```python
mode_index = int(results.mode_indices[0])
separate_plots["mode_shapes"][mode_index]
```

All magnetic bearings are always shown in the mode-shape figures. The selected bearing is highlighted, while the remaining bearings provide positional context. The modal-residue calculation itself still uses only the selected magnetic bearing as the actuator reference.

A single mode shape can also be generated directly:

```python
mode_index = int(results.mode_indices[0])

fig_mode = results.plot_mode_shape(
    mode=mode_index,
)
fig_mode.show()
```

Compatibility regions can optionally be added behind the mode shape:

```python
fig_mode = results.plot_mode_shape(
    mode=mode_index,
    show_compatibility=True,
    background_opacity=0.20,
)
fig_mode.show()
```

## 3.8 Accessing the results of other magnetic bearings

Because the modal residue depends on the actuator position, each magnetic bearing has its own `AmbNonCollocationResults` object.

The results calculated previously can be accessed through the bearing tags:

In [7]:
results_by_tag.keys()
second_bearing = magnetic_bearings[1]
second_results = results_by_tag[second_bearing.tag]

The same methods used for the first AMB are available for `second_results`:

In [8]:
second_table = second_results.modal_residue_table()
display(second_table)

fig = second_results.plot()
fig.show()

,Mode,Frequency (Hz),Actuator amplitude,Sensor amplitude,Modal residue,Normalized residue,Classification
0,1,8.708849,0.388457,0.374234,0.145374,0.374234,Same modal sign
1,2,8.708849,0.388457,0.374234,0.145374,0.374234,Same modal sign
2,3,10.958039,0.798196,0.833793,0.665531,0.833793,Same modal sign
3,4,10.958039,0.798196,0.833793,0.665531,0.833793,Same modal sign
4,5,107.050978,0.456498,0.539620,0.246335,0.539620,Same modal sign
5,6,107.051027,0.456498,0.539620,0.246335,0.539620,Same modal sign
6,7,401.918563,0.040740,0.145190,0.005915,0.145190,Same modal sign
7,8,401.918563,0.040740,0.145190,0.005915,0.145190,Same modal sign


The selected AMB changes the actuator reference used in the modal-residue calculation, while the other AMBs continue to appear as contextual information in the mode-shape visualizations.

## 3.9 Interpretation and limitations

The modal non-collocation analysis provides a screening tool for sensor placement. It identifies positions with strong modal participation, opposite modal signs, or low participation for each retained lateral mode.

The analysis is currently restricted to zero rotational speed, where the lateral eigenvectors can be phase-aligned and represented by real projected mode shapes.

The results should be interpreted together with the time-response and closed-loop analyses presented in the following sections. A favorable modal-residue classification does not replace the evaluation of stability, sensitivity, control effort, actuator saturation, or dynamic performance.

# Section 4: Evaluating Time Response

Active Magnetic Bearing (AMB) systems are inherently coupled, closed-loop systems where the control forces directly depend on the dynamic states of the rotor. To simulate this behavior accurately and efficiently, ROSS utilizes the `AmbTimeResponse` class. 

Instead of relying solely on step-by-step numerical integration at every time step, this class constructs a comprehensive, unified **State-Space Model** in the modal domain. This global model fully encapsulates the rotor structural dynamics, the actuator magnetic forces, and the feedback controller transfer functions.

## 4.1 State-Space Formulation

The coupled system is formulated by combining three fundamental sets of equations:

1. **Rotor Dynamics (Physical Domain):**
$$M \ddot{q}(t) + \left(C + \Omega G\right) \dot{q}(t) + K q(t) = F_{ext}(t) + F_{amb}(t)$$

2. **Magnetic Actuator Force Linearization:**
$$F_{amb}(t) = -K_s \cdot q_{amb}(t) + K_i \cdot i(t)$$

3. **Controller Dynamics (State-Space Representation):**
$$\dot{x}_c(t) = A_c x_c(t) + B_c e(t)$$
$$i(t) = C_c x_c(t) + D_c e(t)$$

By projecting the rotor equations into the modal domain using the modal matrix $\Phi$ ($q = \Phi \eta$) and coupling them with the controller states, ROSS builds a single consolidated closed-loop state vector $z = [\eta^T, \dot{\eta}^T, x_c^T]^T$. The entire system is described by the following linear state-space equations:

$$\dot{z}(t) = A_{cl} z(t) + B_{cl} u(t)$$
$$y(t) = C_{cl} z(t) + D_{cl} u(t)$$

The global block-structured closed-loop matrices are defined as follows:

$$A_{cl} = \begin{bmatrix} 0_{m \times m} & I_{m \times m} & 0_{m \times n} \\ M_m^{-1}\Phi^T\left(\phi R^T (-K_x - K_i D_c)R\phi^T - K\right)\Phi & -M_m^{-1}C_m & M_m^{-1}\Phi^T\phi R^T K_i C_c \\ -B_c R \phi^T \Phi & 0_{n \times m} & A_c \end{bmatrix}$$

$$B_{cl} = \begin{bmatrix} 0_{m \times n_c} & 0_{m \times N} \\ 0_{m \times n_c} & M_m^{-1}\Phi^T \\ -B_c R \phi^T \phi R^T & 0_{n \times N} \end{bmatrix}$$

$$C_{cl} = \begin{bmatrix} \Phi & 0_{N \times m} & 0_{N \times n} \\ \phi^T \Phi & 0_{n_c \times m} & 0_{n_c \times n} \\ R \phi^T \Phi & 0_{n_c \times m} & 0_{n_c \times n} \\ R^T(-K_x - K_i D_c)R \phi^T \Phi & 0_{n_c \times m} & R^T K_i C_c \\ (-K_x - K_i D_c)R \phi^T \Phi & 0_{n_c \times m} & K_i C_c \\ -D_c R \phi^T \Phi & 0_{n_c \times m} & C_c \end{bmatrix}$$

$$D_{cl} = \begin{bmatrix} 0_{N \times n_c} & 0_{N \times N} \\ 0_{n_c \times n_c} & 0_{n_c \times N} \\ 0_{n_c \times n_c} & 0_{n_c \times N} \\ 0_{n_c \times n_c} & 0_{n_c \times N} \\ 0_{n_c \times n_c} & 0_{n_c \times N} \\ 0_{n_c \times n_c} & 0_{n_c \times N} \end{bmatrix}$$

Where:
* $m$ represents the number of dynamic modes retained after modal reduction.
* $n$ indicates the total dimension of the controller's internal state vector.
* $N$ determines the total number of physical degrees of freedom (DoF) of the rotor.
* $n_c$ expresses the number of coordinated control channels for the magnetic bearings (equal to twice the number of magnetic bearings on the rotor).
* $M_m$ and $C_m$ are the modal mass and damping matrices, calculated respectively by $\Phi^T M \Phi$ and $\Phi^T (C + \Omega G) \Phi$. The damping matrix includes the gyroscopic coupling $\Omega G$ at the rotor speed $\Omega$, as in the frequency-domain analyses. The speed is constant in this formulation; for a varying speed, use `method="newmark"`.
* $K$ is the structural physical stiffness matrix of the rotor.
* $\phi$ defines the geometric coupling matrix that locates the bearing nodes within the physical degrees of freedom of the global system.
* $R$ establishes the rotation matrix from the $x$, $y$ axes to the control axes of each magnetic bearing, rotated by its `sensors_axis_rotation`.
* $K_x$ and $K_i$ represent diagonal matrices built from the linear position stiffness ($K_s$) and current stiffness ($K_i$) coefficients of the magnetic forces. Because $K_s < 0$, the term $-K_x$ adds a destabilizing (negative) stiffness at the actuator nodes.
* $A_c, B_c, C_c, D_c$ constitute the dynamic state-space matrices associated with the transfer function of the designed controller. The sensor gain `k_sense` scales the controller input and the amplifier gain `k_amp` scales its output, so that $e$ is the displacement error and $i$ the coil current in amperes.

## 4.2 The `weight` Parameter

When running a time response simulation via `rotor.run_time_response(...)`, you can specify the option `weight=True`. 
* **How it works:** Setting `weight=True` automatically calculates and adds the static gravitational forces ($M \cdot g$) acting on the rotor elements to the external force vector ($F_{ext}$). 
* **Engineering relevance:** This is crucial for AMB systems, as it simulates the static load that the magnetic bearings must overcome to levitate the shaft. It allows you to observe the steady-state bias currents and position offsets required to maintain the rotor centered under gravity.

> **Note on Backwards Compatibility:** The legacy step-by-step numerical integration via the Newmark method can still be explicitly invoked by setting `method="newmark"`. Legacy scripts and workflows utilizing this method remain fully supported and functional.

In [9]:
# 1. Create the rotor with the AMB element
rotor = rs.rotor_example_amb_complex_controllers()
rotor.plot_rotor(nodes=999).show()

In [10]:
# Define simulation parameters
speed = 500.0  # rad/s
t = np.arange(0, 10, 0.001)  # Time vector
F = np.zeros((len(t), rotor.ndof))  # External force vector

# 2. Run time response using the new state-space default approach with gravity included
response = rotor.run_time_response(speed, F, t, weight=True)

# 3. Plot time response for a physical node using a Probe
obs_node = rotor.disk_elements[0].n  # disk node
probe_y = rs.Probe(node=obs_node, angle=np.pi / 2)
fig = response.plot_1d(probe=[probe_y])
fig.show()

# 4. Use the specialized automatic AMB plotting tool for displacements
# (n_points keeps 2000 of the 10,000 samples of each curve, to keep the page light)
response.plot_amb_disps(n_points=2000).show()

Simulation time: 2.08 seconds


# Section 5: Collecting Forces and Control Currents

The `run_time_response` method returns an instance of the `AmbTimeResponseResults` class. This dedicated object acts as a central repository for all internal tracking data related to the magnetic bearings.

## 5.1 Automated Plotting Methods
The results object provides high-level automated plotting methods designed to visualize key control loop variables across all bearings simultaneously:
* `response.plot_amb_disps()`: Plots the displacement profiles.
* `response.plot_amb_currents()`: Plots the control currents over time.
* `response.plot_amb_forces()`: Plots the total magnetic forces applied to the rotor.

## 5.2 Accessing Raw Arrays and Matrix Ordering Rules
If you need to perform custom post-processing, optimization, or alternative plotting configurations, you can access the raw NumPy arrays stored as attributes inside the results object:
* `response.x_amb`: Bearing displacements matrix.
* `response.I`: Control currents matrix.
* `response.F_x`: Total magnetic bearing forces matrix along the coordinate axes.

### Data Structure and Indexing Conventions:
When inspecting these raw matrices, the columns follow a strict geometric and layout convention:
1. **Axis Arrangement:** Within the data columns for any given bearing, the **$x$-axis data always precedes the $y$-axis data**. For instance, column `2*i` holds the $x$ response, and column `2*i + 1` holds the $y$ response.
2. **Node Ordering:** Bearings appear along the columns in the exact spatial sequence they occupy on the rotor shaft. Bearings positioned at **lower node numbers appear first** (from left to right in the matrix columns).

For example, in a system with two magnetic bearings located at Node 11 and Node 39:
* Column 0: Node 11 ($x$-axis)
* Column 1: Node 11 ($y$-axis)
* Column 2: Node 39 ($x$-axis)
* Column 3: Node 39 ($y$-axis)

In [11]:
# 1. Use the automatic plotting methods built into AmbTimeResponseResults
response.plot_amb_currents(n_points=2000).show()
response.plot_amb_forces(n_points=2000).show()

# 2. Extract and inspect raw data vectors directly
raw_currents = response.I
raw_forces = response.F_x
raw_displacements = response.x_amb

print("Shape of the raw current matrix (Time steps x Channels):", raw_currents.shape)
print(
    "\nFirst 5 time steps of raw currents for all AMB axes (ordered by node, x before y):"
)
print(raw_currents[:5, :])

print("\nFirst 5 time steps of raw magnetic forces:")
print(raw_forces[:5, :])

Shape of the raw current matrix (Time steps x Channels): (10000, 4)

First 5 time steps of raw currents for all AMB axes (ordered by node, x before y):
[[0.         0.         0.         0.        ]
 [0.05476333 0.05476333 0.09736979 0.09736979]
 [0.16288844 0.16288844 0.13354899 0.13354899]
 [0.27113666 0.27113666 0.16622477 0.16622477]
 [0.31934007 0.31934007 0.19890555 0.19890555]]

First 5 time steps of raw magnetic forces:
[[0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [1.80948177e-13 7.46915752e+00 1.89605563e-12 1.44918220e+01]
 [1.34647037e-12 2.15628549e+01 7.71444640e-12 1.88741479e+01]
 [4.09090668e-12 3.49690982e+01 1.49569544e-11 2.27897434e+01]
 [8.16767035e-12 3.92025693e+01 2.23197509e-11 2.67571750e+01]]


# Section 6: Sensitivity Analysis (ISO 14839)

## 6.1 Definition and ISO Standard

The sensitivity function $S(j\omega)$ determines the robustness of the control system. According to **ISO 14839-3**, the sensitivity is the ratio of the sensor output to a disturbance added to the sensor signal. It represents how much a disturbance is amplified by the feedback loop.

$$
S(j\omega) = \frac{1}{1 + G(j\omega)H(j\omega)}
$$

The ISO standard classifies the stability margin based on the **peak sensitivity value ($S_{max}$)**:

- **Zone A ($S_{max} < 3.0$):** Unrestricted operation (approx 9.5 dB).
- **Zone B ($3.0 < S_{max} < 4.0$):** Restricted operation.
- **Zone C ($S_{max} > 4.0$):** Evaluation required, potential instability.

## 6.2 Simulation Procedure in ROSS

To calculate the sensitivity function, ROSS performs a specific time-domain simulation:

1. A **Logarithmic Chirp Signal** (sweeping sine wave) is injected as a disturbance into the control loop at the AMB sensor location.
2. The system response (displacement) is recorded.
3. A Frequency Response Function (FRF) is computed between the **Output (Disturbed Signal)** and the **Input (Excitation Signal)**.

In [12]:
# Run sensitivity analysis
# This automatically performs the chirp injection and FFT computation
rotor = rs.rotor_example_amb_simple()
f_max = 150  # Hz, highest chirp frequency
sensitivity_results = rotor.run_amb_sensitivity(
    speed=0,
    t_max=10,  # Duration must be long enough for the chirp
    dt=1e-5,  # Time step
    disturbance_amplitude=1e-5,
    disturbance_min_frequency=0.001,  # Hz
    disturbance_max_frequency=f_max,
    amb_tags=["AMB_1"],
    verbose=0,
);

## 6.3 The `SensitivityResults` Object

The `run_amb_sensitivity` method returns a `SensitivityResults` object. Below is a summary of its attributes and methods.

### Attributes Table

| Attribute | Type | Description |
| --- | --- | --- |
| `sensitivities` | `dict` | Dictionary containing the complex Sensitivity FRF ($S(j\omega)$) for each AMB and axis ($x$, $y$). |
| `sensitivities_abs` | `dict` | The magnitude (absolute value) of the sensitivity function. |
| `sensitivities_phase` | `dict` | The phase angle of the sensitivity function. |
| `sensitivities_frequencies` | `np.ndarray` | The frequency vector (in Hz) corresponding to the sensitivity arrays. |
| `max_abs_sensitivities` | `dict` | The peak sensitivity value ($S_{max}$) for each AMB/axis. Used for ISO classification. |
| `sensitivity_run_time_results` | `dict` | Contains raw time-domain arrays: `excitation_signal` (chirp), `disturbed_signal` (input to controller), and `sensor_signal`. |
| `sensitivity_compute_dofs` | `dict` | Mapping of AMB tags to their corresponding Degree of Freedom indices. |

### Accessing Results Data

The attributes within the `SensitivityResults` object (such as `max_abs_sensitivities`, `sensitivities`, `sensitivities_abs`, and `sensitivities_phase`) are organized as **nested dictionaries**. This structure allows you to easily retrieve data for a specific magnetic bearing and a specific axis.

**Structure Hierarchy:**

1. **First Level Key:** The AMB tag (string), exactly as defined when creating the `MagneticBearingElement`.
2. **Second Level Key:** The axis (string), either `"x"` or `"y"`.
3. **Value:** The corresponding data (scalar for max sensitivity, or array for FRFs).

**Visual Representation:**

```python
sensitivity_results.max_abs_sensitivities = {
    "AMB_Tag_1": {
        "x": <scalar_value>,
        "y": <scalar_value>
    },
    "AMB_Tag_2": {
        "x": <scalar_value>,
        "y": <scalar_value>
    }
    # ...
}
```

**Example: Retrieving Maximum Sensitivity**

The following example demonstrates how to programmatically access the maximum absolute sensitivity ($S_{max}$) for a specific axis of a specific bearing to check against ISO limits.

In [13]:
# Assume 'sensitivity_results' is the object returned by rotor.run_amb_sensitivity()

# 1. Define the target Bearing Tag and Axis
target_amb_tag = "AMB_1"  # Must match the 'tag' used in MagneticBearingElement
target_axis = "x"

# 2. Access the nested dictionary
s_max = sensitivity_results.max_abs_sensitivities[target_amb_tag][target_axis]

print(f"Peak Sensitivity for {target_amb_tag} ({target_axis}-axis): {s_max:.4f}")

# 3. Check against ISO 14839-3 Zone A limit
if s_max < 3.0:
    print("Result: Zone A (Unrestricted Operation)")
else:
    print("Result: Zone B or C (Evaluation required)")

Peak Sensitivity for AMB_1 (x-axis): 18.9984
Result: Zone B or C (Evaluation required)


### Plotting Methods

**`.plot()`**

Displays the Bode plot (Magnitude and Phase) of the sensitivity function. This is the primary tool to check compliance with ISO 14839.

In [14]:
# Plot Bode diagram of Sensitivity
fig_bode = sensitivity_results.plot(
    frequency_units="Hz",
    magnitude_scale="decibel",  # Useful to check margins in dB
    xaxis_type="log",
)
# The FFT reaches the Nyquist frequency (50 kHz); keep the band the chirp excites
for trace in fig_bode.data:
    in_band = trace.x <= f_max
    trace.x, trace.y = trace.x[in_band], trace.y[in_band]
fig_bode.update_xaxes(autorange=True)
fig_bode.show()

**`.plot_time_results()`**

Displays the raw time-domain signals used to calculate the sensitivity. This is useful for debugging to ensure the chirp signal was applied correctly and the system remained stable during the test.

In [15]:
# Plot the Chirp injection and system response
# Show 10,000 of the 10**6 samples of each signal to keep the page light
sensitivity_results.plot_time_results(n_points=10_000).show()

# Section 7: Implementing Complex Controllers (Cascade)

ROSS supports arbitrary transfer functions using the `python-control` library. You can combine controllers (e.g., in series/cascade) by multiplying their transfer functions.

**Example:** A PID controller cascaded with a **Lead Compensator** (to improve phase margin).

In [16]:
# 1. Define the Laplace variable 's'
s = rs.MagneticBearingElement.s

# 2. Define PID Controller
Kp, Ki, Kd = 1500, 100, 10
# Standard PID formulation with a derivative filter (N) usually handled internally or explicitly:
# Here we use a pure PID for simplicity, or construct manual TF
TF_PID = Kp + Ki / s + Kd * s

# 3. Define Lead Controller
# Lead: (alpha * T * s + 1) / (T * s + 1), where alpha > 1
alpha = 3.0
T = 0.001
TF_Lead = (alpha * T * s + 1) / (T * s + 1)

# 4. Cascade (Series) Connection
# In frequency domain, series connection is multiplication
TF_Combined = TF_PID * TF_Lead

# 5. Instantiate AMB with the complex controller
amb_cascade = rs.MagneticBearingElement(
    n=4,
    g0=1e-3,
    i0=1.0,
    ag=1e-4,
    nw=200,
    controller_transfer_function=TF_Combined,  # Pass the object here
    tag="AMB_Cascade",
)

ROSS handles the discretization and state-space conversion of `TF_Combined` automatically during the simulation.

It is worth noting that, if necessary, the `MagneticBearingElement` can still be defined by specifying only the `kp_pid`, `ki_pid`, and `kd_pid` parameters.

## 7.1 Auxiliary Methods for Defining and Evaluating Transfer Functions

### Conventions and returned types

All “controller builder” functions return **python-control transfer functions** (`control.TransferFunction`), except where noted:

* `pid(...)` → `TransferFunction`
* `lead_lag(...)` → `TransferFunction`
* `second_order(...)` → `TransferFunction`
* `low_pass_filter(...)` → `TransferFunction`
* `notch_filter(...)` → `TransferFunction`
* `lqg(...)` → `TransferFunction` (converted from state-space to TF internally)
* `combine(*args)` → product of transfer functions (series connection)

In [17]:
from ross.bearings.magnetic.amb_controllers import *

Below, each of the auxiliary methods introduced above is described in detail.

**`pid(k_p, k_i, k_d, n_f=10000)`**

Builds a PID controller with a **filtered derivative** term.

* `k_p` *(float)*: proportional gain
* `k_i` *(float)*: integral gain
* `k_d` *(float)*: derivative gain
* `n_f` *(float, optional)*: derivative filter “corner” scaling (higher → closer to ideal derivative without filtering)

In [18]:
C = pid(k_p=3.0, k_i=5.0, k_d=0.02, n_f=300)

**`lqg(A, B, C, Q_lqr, R_lqr, Q_kalman, R_kalman)`**

Builds an **LQG controller** (LQR state feedback + Kalman filter) and returns it as a transfer function.

* `A`, `B`, `C`: system matrices (array-like). Converted to `float` numpy arrays.
* `Q_lqr`, `R_lqr`: LQR weighting matrices.
* `Q_kalman`, `R_kalman`: process/measurement noise covariances for the Kalman filter design.

**Notes / tips**

* Dimensions must match python-control expectations, where `n`, `m`, and `p` are, respectively, the number of states, controls, and outputs.:
  * `A` is `(n, n)`, `B` is `(n, m)`, `C` is `(p, n)`
  * `Q_lqr` is `(n, n)`, `R_lqr` is `(m, m)`
  * `Q_kalman` is `(n, n)`, `R_kalman` is `(p, p)`

In [19]:
A = [[0, 1], [-2, -0.5]]
B = [[0], [1]]
C = [[1, 0]]

Q_lqr = np.diag([10, 1])
R_lqr = np.array([[1]])

Q_kalman = np.diag([0.1, 0.1])
R_kalman = np.array([[0.5]])

C_lqg = lqg(A, B, C, Q_lqr, R_lqr, Q_kalman, R_kalman)

**`lead_lag(tau, alpha, k=1.0)`**

Creates a first-order lead/lag compensator:

$$C(s)=\frac{k(\tau s + 1)}{\alpha \tau s + 1}$$

* `tau` *(float)*: time constant
* `alpha` *(float)*: pole/zero separation factor
  * **Lead** is typically `0 < alpha < 1` (pole farther left than zero)
  * **Lag** is typically `alpha > 1` (pole closer to origin than zero)
* `k` *(float, optional)*: gain multiplier

In [20]:
C_lead = lead_lag(tau=0.02, alpha=0.1, k=2.0)
C_lag = lead_lag(tau=0.5, alpha=5.0, k=1.0)

**`second_order(b2, b1, b0, a1, a0)`**

Creates a generic second-order transfer function:

$$H(s)=\frac{b_2 s^2 + b_1 s + b_0}{s^2 + a_1 s + a_0}$$

In [21]:
H = second_order(b2=1, b1=0.2, b0=10, a1=0.5, a0=25)

**`low_pass_filter(w_c, k=1.0)`**

First-order low-pass filter:

$$F(s)=\frac{k\,\omega_c}{s + \omega_c}$$

* `w_c` *(float)*: cutoff frequency in rad/s
* `k` *(float, optional)*: DC gain factor

In [22]:
F = low_pass_filter(w_c=200, k=1.0)

**`notch_filter(w_n, zeta_z, zeta_p, k=1.0)`**

Second-order notch filter (zeros and poles around the same natural frequency):

$$N(s)=k\frac{s^2 + 2\zeta_z\omega_n s + \omega_n^2}{s^2 + 2\zeta_p\omega_n s + \omega_n^2}$$

* `w_n` *(float)*: notch center frequency (rad/s)
* `zeta_z` *(float)*: zero damping (controls notch depth/shape)
* `zeta_p` *(float)*: pole damping (controls bandwidth/sharpness)
* `k` *(float, optional)*: gain multiplier

In [23]:
N = notch_filter(w_n=120, zeta_z=0.01, zeta_p=0.2, k=1.0)

**`combine(*args)`**

Multiplies transfer functions in series.

* `*args`: any number of `control.TransferFunction` objects

In [24]:
C = combine(
    pid(2, 1, 0.02, n_f=200),
    lead_lag(0.03, 0.1),
    notch_filter(120, 0.02, 0.2),
    low_pass_filter(400),
)

**`plot_frequency_response(*systems, **kwargs)`**

Plots magnitude (dB) and phase (degrees) for one or more systems using Plotly.

* `*systems`: one or more LTI systems compatible with `ct.frequency_response(...)` (`control.TransferFunction` or `control.StateSpace`)
* `w_min` *(float, default=1e-2)*: minimum frequency (rad/s)
* `w_max` *(float, default=1e3)*: maximum frequency (rad/s)
* `n_points` *(int, default=1000)*: number of log-spaced frequency points
* `title` *(str, default="Frequency Response")*: plot title
* `legends` *(list[str] | None)*: legend labels; must match number of systems

In [25]:
plot_frequency_response(
    C,
    N,
    C * N,
    legends=["Controller", "Filter", "Combined"],
    w_min=1e-1,
    w_max=1e5,
    n_points=1500,
    title="Bode (Controller / Filter / Combined)",
)

# Section 8: Equivalent Stiffness Calculation

For linear frequency-domain analyses (e.g., `run_modal`, `run_campbell`, `run_ucs`), ROSS cannot evaluate the discrete time-domain controller states directly. Instead, the `MagneticBearingElement` computes **frequency-dependent equivalent coefficients** that represent the closed-loop AMB behavior in the form of an equivalent stiffness and damping.

With the updated implementation, the **displacement sensor gain** (`k_sense`, in V/m) and the **power amplifier gain** (`k_amp`, in A/V, coil current per unit of controller output) are explicitly included in the loop gain used to build these equivalent coefficients. The same gains are applied in the time-domain simulations (`AmbTimeResponse` and the Newmark controller), so all analyses use the same loop gain.

Internally, the element performs:

1. Build (or retrieve) the continuous-time controller transfer function $C(s)$ (PID with derivative filter or a custom transfer function).
2. Evaluate its frequency response $C(j\omega)$ over a frequency grid $\omega$ (rad/s).
3. Split the response into real and imaginary parts:
   $
   C(j\omega)=\Re\{C(j\omega)\} + j \, \Im\{C(j\omega)\}
   $
4. Map these parts into equivalent stiffness and damping using the electromagnetic constants:
   * $K_s$: negative electromagnetic stiffness (from force linearization)
   * $K_i$: current-to-force gain (from force linearization)

### Equivalent stiffness

The equivalent stiffness includes the open-loop negative stiffness $K_s$ plus the real part of the controller contribution scaled by the sensor and amplifier gains:

$$K_{eq}(\omega) = K_s + K_i \, k_{amp} \, k_{sense} \, \Re\{C(j\omega)\}$$

### Equivalent damping

The equivalent damping is formed from the imaginary part of the controller contribution, scaled by the same gains and divided by $\omega$:

$$
C_{eq}(\omega) = \frac{K_i \, k_{amp} \, k_{sense}}{\omega} \, \Im\{C(j\omega)\}
$$

### Notes on implementation details

* In the code, $C(j\omega)$ is obtained from `control.frequency_response()`, stored as `Hjw`, and then:

  * `C_real = Hjw.real`
  * `C_imag = Hjw.imag`
* The arrays are computed as:
  $$
  k_{eq} = K_s + K_i \, k_{amp} \, k_{sense} \, C_{real}
  \qquad
  c_{eq} = K_i \, k_{amp} \, k_{sense} \, C_{imag} \, \frac{1}{\omega}
  $$
* After that, the element transforms the isotropic equivalent coefficients ${k_{eq}, c_{eq}}$ into the rotor $x\text{–}y$ coordinates using the rotation defined by `sensors_axis_rotation`, producing $k_{xx}, k_{xy}, k_{yx}, k_{yy}$ and $c_{xx}, c_{xy}, c_{yx}, c_{yy}$.

These frequency-dependent arrays are stored in the element. During calls like `run_modal` (at a given speed), ROSS interpolates the coefficients at the required excitation frequencies to assemble the global stiffness and damping matrices for the rotor model.

# Section 9: Operating Point Under a Static Load

By default, `ki` and `ks` are the nominal values of ISO 14839-1: centered rotor and no control current. A bearing that carries a static load works with the control current $i_{c0}$ that holds it, and the slopes of the magnetic force at that point differ from the nominal ones. Passing `static_load=(Fx, Fy)`, the static force the bearing exerts on the rotor (N), linearizes each control axis at that operating point:

- with integral action the rotor stays centered; while both coils conduct, $K_i$ is unchanged and $K_s = K_{s,0} \left[1 + (i_{c0}/i_0)^2\right]$;
- when $|i_{c0}| > i_0$ the opposite coil stops conducting and a single electromagnet holds the load;
- without integral action the rotor moves until the static controller force holds the load, and both coefficients change.

The load is projected on the control axes (`sensors_axis_rotation`), so each axis can have its own operating point. `static_current` and `static_displacement` report it.

The static reactions of `run_static` give the load each bearing carries:

In [26]:
rotor = rs.rotor_example_amb_complex_controllers()
rotor.run_static()

In [27]:
{node: round(float(force), 2) for node, force in rotor.bearing_forces_nodal.items()}

{'node_11': 32.53, 'node_39': 25.23}

Each magnetic bearing is rebuilt with its reaction as `static_load`, and the coefficients are compared with the nominal ones:

In [28]:
def with_static_load(bearing, load):
    """Rebuild a magnetic bearing that holds the static force (Fx, Fy)."""
    return rs.MagneticBearingElement(
        n=bearing.n,
        g0=float(bearing.g0),
        i0=bearing.i0,
        ag=float(bearing.ag),
        nw=float(bearing.nw),
        alpha=float(bearing.alpha),
        k_amp=bearing.k_amp,
        k_sense=bearing.k_sense,
        controller_transfer_function=bearing.get_analog_controller(),
        sensors_axis_rotation=float(bearing.sensors_axis_rotation),
        sensor_node=bearing.sensor_node,
        static_load=load,
        tag=bearing.tag,
    )


loaded_bearings = [
    with_static_load(bearing, (0.0, rotor.bearing_forces_nodal[f"node_{bearing.n}"]))
    for bearing in rotor.bearing_elements
]
loaded_rotor = rs.Rotor(rotor.shaft_elements, rotor.disk_elements, loaded_bearings)

for nominal, loaded in zip(rotor.bearing_elements, loaded_bearings, strict=True):
    ki = np.atleast_1d(loaded.ki)
    ks = np.atleast_1d(loaded.ks) / 1e3
    print(f"Node {loaded.n}: static load {loaded.static_load[1]:.1f} N")
    print(f"  control current per axis: {np.round(loaded.static_current, 3)} A")
    print(f"  ki: {float(nominal.ki):.2f} N/A -> {np.round(ki, 2)} N/A")
    print(f"  ks: {float(nominal.ks) / 1e3:.1f} kN/m -> {np.round(ks, 1)} kN/m")

Node 11: static load 32.5 N
  control current per axis: [0.206 0.206] A
  ki: 111.47 N/A -> [111.47 111.47] N/A
  ks: -258.0 kN/m -> [-269. -269.] kN/m
Node 39: static load 25.2 N
  control current per axis: [0.16 0.16] A
  ki: 111.47 N/A -> [111.47 111.47] N/A
  ks: -258.0 kN/m -> [-264.6 -264.6] kN/m


The weight of this rotor needs a small control current, and the coefficients barely change. To show how the effect grows, the analysis is repeated with the static loads multiplied by a factor:

- factor 0 gives the nominal coefficients (no load);
- factor 1 is the weight of the rotor, the real condition of this test rig;
- larger factors stand for an additional radial load, such as a heavier disk or a process load.

The vibration modes are taken from the closed-loop poles of the state-space model of `AmbTimeResponse`, in which each magnetic bearing is its controller (with its own states) acting through `ki` and `ks`. `run_modal` is not used here: it evaluates the bearing coefficients at a frequency equal to the rotor speed, and at zero speed the integral action of this controller makes the bearings behave as almost rigid supports, which hides the effect of `ks`.

For each case the table lists the control current and `ks` of both bearings, and the natural frequency and damping ratio of the first three closed-loop vibration modes:

In [29]:
import pandas as pd
from ross.bearings.magnetic.amb_time_response import AmbTimeResponse


def closed_loop_modes(rotor, n_modes=3, max_damping=0.7):
    """Natural frequency (Hz) and damping ratio of the first closed-loop vibration modes.

    The magnetic bearings do not support the axial and torsional rigid-body motion,
    so those poles are at zero and are left out, as are poles with a damping ratio
    of max_damping or more. Each lateral mode appears in two directions with the
    same frequency, so only one of them is kept.
    """
    sim = AmbTimeResponse(rotor, t=np.linspace(0, 1e-3, 3), speed=0, weight=False)
    sim.process_rotor()
    sim.setup_modal_domain()
    sim.build_matrices()
    poles = np.linalg.eigvals(sim.A_cl)
    poles = poles[(poles.imag > 1e-6) & (np.abs(poles) > 2 * np.pi * 0.1)]
    modes = sorted(
        zip(np.abs(poles) / (2 * np.pi), -poles.real / np.abs(poles), strict=True)
    )
    distinct = []
    for frequency, damping in modes:
        if damping >= max_damping:
            continue
        if distinct and np.isclose(frequency, distinct[-1][0], rtol=1e-3):
            continue
        distinct.append((frequency, damping))
    return distinct[:n_modes]


rows = []
for factor in (0, 1, 2, 3, 4):
    bearings = [
        with_static_load(
            bearing, (0.0, factor * rotor.bearing_forces_nodal[f"node_{bearing.n}"])
        )
        for bearing in rotor.bearing_elements
    ]
    model = rs.Rotor(rotor.shaft_elements, rotor.disk_elements, bearings)
    row = {"load / weight": factor}
    for bearing in bearings:
        row[f"ic0 node {bearing.n} (A)"] = np.max(np.abs(bearing.static_current))
        row[f"ks node {bearing.n} (kN/m)"] = np.atleast_1d(bearing.ks)[0] / 1e3
    for mode, (frequency, damping) in enumerate(closed_loop_modes(model), start=1):
        row[f"mode {mode} (Hz)"] = frequency
        row[f"zeta {mode}"] = damping
    rows.append(row)

pd.DataFrame(rows).set_index("load / weight").round(3)

,ic0 node 11 (A),ks node 11 (kN/m),ic0 node 39 (A),ks node 39 (kN/m),mode 1 (Hz),zeta 1,mode 2 (Hz),zeta 2,mode 3 (Hz),zeta 3
load / weight,,,,,,,,,,
0,0.000,-258.026,0.00,-258.026,55.109,0.602,88.600,0.104,162.604,0.550
1,0.206,-269.011,0.16,-264.637,53.809,0.608,88.664,0.104,162.425,0.551
2,0.413,-301.965,0.32,-284.473,49.544,0.630,88.841,0.102,161.891,0.554
3,0.619,-356.890,0.48,-317.533,40.361,0.658,89.094,0.100,161.008,0.560
4,0.825,-433.784,0.64,-363.816,29.162,0.497,89.379,0.097,159.790,0.568


$|K_s|$ grows with the square of the control current: about 4 % at the weight of the rotor and about 68 % at four times the weight in the bearing at node 11. The mode governed by the bearings is the most affected: it drops from about 55 Hz to about 54 Hz at the weight of the rotor, and to about 29 Hz at four times the weight, with less damping. The other two modes (near 89 Hz and 162 Hz) change by less than 2 %. At $i_{c0} = i_0$ (about 4.8 times the weight for the bearing at node 11) the opposite coil stops conducting, a single electromagnet holds the load, and $|K_s|$ is twice its nominal value. Fringing, iron reluctance and magnetic saturation are not modeled.